# UATimer — Measurement & Analysis Notebook

This notebook is the **analysis pipeline** for the UATimer evaluation. It
turns **raw per-run measurements** into the aggregated tables, statistics,
and figures used in the manuscript.

> **No data is fabricated here.** The notebook reads a raw CSV that *you*
> populate from real hardware measurements (Monsoon power monitor / external
> meter, timestamp logs). To let you run the notebook before your data is
> ready, a clearly-labelled **synthetic demo** mode fills the tables with
> random illustrative numbers — these are *not* the paper's results and must
> be replaced. Set `USE_SYNTHETIC_DEMO = False` once `data/raw_runs.csv`
> exists.

**What it produces**
1. Schema validation of the raw per-run data.
2. Per-(workload, policy) mean, standard deviation, and 95% confidence interval.
3. Energy-reduction percentages (vs. Baseline / Fixed / LR) with propagated uncertainty.
4. Significance tests (Wilcoxon signed-rank) between UATimer and each comparator.
5. Figures: energy–delay operating points and per-workload energy with error bars.
6. LaTeX for Table V (results) and Table VI (reductions), generated from the data.

**Addresses review items:** M6 (statistics / raw data), M8 (baselines),
and the AR/VR quality reporting.

In [ ]:
# --- Configuration ---------------------------------------------------------
from pathlib import Path

# Set to False and provide data/raw_runs.csv with your real measurements.
USE_SYNTHETIC_DEMO = True

# Repository-relative paths. Adjust if you move the notebook.
ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_CSV = ROOT / "data" / "raw_runs.csv"
OUT_DIR  = ROOT / "results"
OUT_DIR.mkdir(exist_ok=True)

# Analysis settings
CONF_LEVEL   = 0.95
POLICY_ORDER = ["Baseline", "Fixed", "LR", "UATimer"]
WORKLOADS    = ["Web Browsing", "Video Playback", "AI Inference",
                "AR/VR Gaming", "IoT Sensing"]
print("root:", ROOT)
print("raw CSV:", DATA_CSV, "(exists:", DATA_CSV.exists(), ")")
print("synthetic demo:", USE_SYNTHETIC_DEMO)

## 1. Raw data schema

`data/raw_runs.csv` must have **one row per experimental run** with columns:

| column | meaning |
|---|---|
| `platform` | device id (e.g. `phone`, `tablet`, `iot_node`) |
| `workload` | one of the workload names |
| `policy` | one of `Baseline`, `Fixed`, `LR`, `UATimer` |
| `run` | integer run index (1..N) |
| `energy_mj` | measured energy for the run (mJ) |
| `delay_ms` | mean event delay for the run (ms) |
| `frame_drop_pct` | AR/VR only, else empty |
| `jitter_ms` | AR/VR only, else empty |

Record **every individual run** (the manuscript notes ≥10 runs per
condition are needed for dispersion and significance).

In [ ]:
# --- Load or synthesize the raw per-run data -------------------------------
import numpy as np, pandas as pd

REQUIRED_COLS = ["platform","workload","policy","run",
                 "energy_mj","delay_ms","frame_drop_pct","jitter_ms"]

def make_synthetic_demo(n_runs=10, seed=0):
    '''Illustrative random data ONLY. Not the paper's measurements.'''
    rng = np.random.default_rng(seed)
    # Rough per-policy multipliers to give a plausible SHAPE, not real values.
    base = {"Web Browsing":(1200,30),"Video Playback":(1800,25),
            "AI Inference":(2200,28),"AR/VR Gaming":(2500,20),
            "IoT Sensing":(400,12)}
    emul = {"Baseline":1.00,"Fixed":0.82,"LR":0.75,"UATimer":0.73}
    dmul = {"Baseline":1.00,"Fixed":1.6,"LR":1.5,"UATimer":1.45}
    rows=[]
    for wl,(e0,d0) in base.items():
        plat = "iot_node" if wl=="IoT Sensing" else "phone"
        for pol in POLICY_ORDER:
            for r in range(1,n_runs+1):
                e = e0*emul[pol]*(1+rng.normal(0,0.04))
                d = d0*dmul[pol]*(1+rng.normal(0,0.05))
                fd=jt=np.nan
                if wl=="AR/VR Gaming":
                    fd={"Baseline":2.5,"Fixed":3.2,"LR":1.8,"UATimer":0.5}[pol]*(1+rng.normal(0,0.1))
                    jt={"Baseline":3.0,"Fixed":3.5,"LR":2.7,"UATimer":1.2}[pol]*(1+rng.normal(0,0.1))
                rows.append([plat,wl,pol,r,round(e,1),round(d,1),
                             round(fd,2) if fd==fd else np.nan,
                             round(jt,2) if jt==jt else np.nan])
    return pd.DataFrame(rows,columns=REQUIRED_COLS)

if USE_SYNTHETIC_DEMO or not DATA_CSV.exists():
    if not USE_SYNTHETIC_DEMO:
        raise FileNotFoundError(f"{DATA_CSV} not found. Provide it or set USE_SYNTHETIC_DEMO=True.")
    print("*** SYNTHETIC DEMO DATA — NOT MEASUREMENTS. Replace before use. ***")
    df = make_synthetic_demo()
else:
    df = pd.read_csv(DATA_CSV)

missing = set(REQUIRED_COLS) - set(df.columns)
assert not missing, f"raw CSV is missing columns: {missing}"
df["policy"]  = pd.Categorical(df["policy"], POLICY_ORDER, ordered=True)
print(f"{len(df)} runs, {df['workload'].nunique()} workloads, "
      f"{df['policy'].nunique()} policies")
df.head()

## 2. Data-quality checks

Confirms every (workload, policy) cell has the same number of runs and no
missing energy/delay values, so later statistics are well defined.

In [ ]:
# --- Sanity checks ---------------------------------------------------------
counts = df.pivot_table(index="workload", columns="policy",
                        values="run", aggfunc="count", observed=True)
display(counts)

n_per_cell = counts.stack().dropna().unique()
print("runs per cell:", sorted(n_per_cell))
if len(n_per_cell) > 1:
    print("WARNING: unbalanced run counts across cells.")
if (counts.stack() < 10).any():
    print("WARNING: some cells have <10 runs; dispersion/significance will be weak.")
assert df[["energy_mj","delay_ms"]].notna().all().all(), "missing energy/delay values"
print("checks complete")

## 3. Aggregate: mean, standard deviation, 95% CI

The confidence interval uses the Student-*t* distribution, appropriate for
small samples.

In [ ]:
# --- Aggregation with t-based confidence intervals -------------------------
from scipy import stats

def agg_metric(frame, col):
    g = frame.groupby(["workload","policy"], observed=True)[col]
    out = g.agg(["mean","std","count"]).reset_index()
    # t critical value per cell (count may vary)
    tcrit = out["count"].apply(lambda n: stats.t.ppf(0.5+CONF_LEVEL/2, n-1) if n>1 else np.nan)
    out["sem"] = out["std"] / np.sqrt(out["count"])
    out["ci95"] = tcrit * out["sem"]
    return out

energy_stats = agg_metric(df, "energy_mj")
delay_stats  = agg_metric(df, "delay_ms")
energy_stats.round(2).head(8)

In [ ]:
# Pretty 'mean ± ci' tables for energy and delay
def pivot_pm(stats_df, unit):
    def fmt(r): return f"{r['mean']:.0f} ± {r['ci95']:.0f}"
    stats_df = stats_df.copy()
    stats_df["cell"] = stats_df.apply(fmt, axis=1)
    p = stats_df.pivot(index="workload", columns="policy", values="cell")
    return p.reindex(index=[w for w in WORKLOADS if w in p.index])[
        [c for c in POLICY_ORDER if c in p.columns]]

print("Energy (mJ), mean ± 95% CI"); display(pivot_pm(energy_stats,"mJ"))
print("Delay (ms), mean ± 95% CI");  display(pivot_pm(delay_stats,"ms"))

## 4. Energy reduction with propagated uncertainty

For each workload, reduction of UATimer vs. a comparator is
`1 - E_uatimer / E_comparator`. The uncertainty is propagated from the
CI of both means (independent-ratio approximation).

In [ ]:
# --- Reductions vs Baseline / Fixed / LR -----------------------------------
em = energy_stats.set_index(["workload","policy"])

def reduction(wl, comparator):
    a = em.loc[(wl,"UATimer")]; b = em.loc[(wl,comparator)]
    ratio = a["mean"]/b["mean"]
    # relative errors combine in quadrature for a ratio
    rel = np.sqrt((a["ci95"]/a["mean"])**2 + (b["ci95"]/b["mean"])**2)
    red = (1-ratio)*100
    red_ci = ratio*rel*100
    return red, red_ci

rows=[]
for wl in [w for w in WORKLOADS if (w,"UATimer") in em.index]:
    row={"workload":wl}
    for comp in ["Baseline","Fixed","LR"]:
        r,ci = reduction(wl,comp)
        row[f"vs {comp} (%)"]=f"{r:.1f} ± {ci:.1f}"
    rows.append(row)
red_df = pd.DataFrame(rows).set_index("workload")
display(red_df)
print("Range vs Baseline:",
      f"{min(reduction(w,'Baseline')[0] for w in WORKLOADS if (w,'UATimer') in em.index):.1f}"
      f"–{max(reduction(w,'Baseline')[0] for w in WORKLOADS if (w,'UATimer') in em.index):.1f}%")

## 5. Significance tests

Paired Wilcoxon signed-rank test on per-run energy (UATimer vs. each
comparator) when runs are paired by index; falls back to Mann–Whitney U
if the design is unpaired. Small samples give low power — treat p-values
as indicative, and report the test, statistic, and n.

In [ ]:
# --- Wilcoxon signed-rank (paired) energy comparison -----------------------
def paired_energy(wl, pol):
    s = (df[(df.workload==wl)&(df.policy==pol)]
         .sort_values("run")["energy_mj"].to_numpy())
    return s

sig_rows=[]
for wl in [w for w in WORKLOADS if w in df.workload.unique()]:
    ua = paired_energy(wl,"UATimer")
    for comp in ["Baseline","Fixed","LR"]:
        cc = paired_energy(wl,comp)
        n = min(len(ua),len(cc))
        if n < 3:
            sig_rows.append([wl,comp,"n<3","-",n]); continue
        try:
            stat,p = stats.wilcoxon(ua[:n], cc[:n])
            test="Wilcoxon"
        except ValueError:
            stat,p = stats.mannwhitneyu(ua, cc, alternative="two-sided")
            test="MannWhitneyU"
        sig_rows.append([wl,comp,test,f"{p:.4f}",n])
sig_df = pd.DataFrame(sig_rows,columns=["workload","vs","test","p_value","n"])
display(sig_df)
print("Note: with the demo's n per cell, significance is illustrative only.")

## 6. Figures

Energy–delay operating points (normalized to each workload's baseline) and
per-workload energy with 95% CI error bars. Saved to `results/`.

In [ ]:
# --- Figure 1: energy-delay operating points -------------------------------
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6,4.2))
markers={"Baseline":"s","Fixed":"^","LR":"D","UATimer":"o"}
es = energy_stats.set_index(["workload","policy"])
ds = delay_stats.set_index(["workload","policy"])
for wl in [w for w in WORKLOADS if (w,"Baseline") in es.index]:
    e0 = es.loc[(wl,"Baseline"),"mean"]
    xs=[ds.loc[(wl,p),"mean"] for p in POLICY_ORDER]
    ys=[es.loc[(wl,p),"mean"]/e0 for p in POLICY_ORDER]
    ax.plot(xs,ys,color="0.7",lw=0.8,zorder=1)
    for p in POLICY_ORDER:
        ax.scatter(ds.loc[(wl,p),"mean"], es.loc[(wl,p),"mean"]/e0,
                   marker=markers[p], zorder=2,
                   label=p if wl==WORKLOADS[0] else None)
ax.set_xlabel("Mean event delay (ms)"); ax.set_ylabel("Energy / baseline energy")
ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
fig.savefig(OUT_DIR/"fig_energy_delay.pdf"); fig.savefig(OUT_DIR/"fig_energy_delay.png",dpi=150)
plt.show()

In [ ]:
# --- Figure 2: per-workload energy with 95% CI -----------------------------
fig, ax = plt.subplots(figsize=(6.5,4))
wls=[w for w in WORKLOADS if (w,"Baseline") in es.index]
x=np.arange(len(wls)); w=0.2
for i,p in enumerate(POLICY_ORDER):
    means=[es.loc[(wl,p),"mean"] for wl in wls]
    cis  =[es.loc[(wl,p),"ci95"] for wl in wls]
    ax.bar(x+(i-1.5)*w, means, w, yerr=cis, capsize=3, label=p)
ax.set_xticks(x); ax.set_xticklabels([w.split()[0] for w in wls])
ax.set_ylabel("Energy (mJ)"); ax.legend(); ax.grid(axis="y",alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR/"fig_energy_bars.pdf"); fig.savefig(OUT_DIR/"fig_energy_bars.png",dpi=150)
plt.show()

## 7. Export LaTeX tables

Generates `results/table_results.tex` (energy/delay means) and
`results/table_reductions.tex` (reductions), ready to `\input{}` into the
manuscript. When you have ≥10 runs, add the `± CI` columns to the paper.

In [ ]:
# --- Emit LaTeX for the results and reduction tables -----------------------
def latex_results():
    lines=[r"% auto-generated from data/raw_runs.csv by notebooks/uatimer_analysis.ipynb",
           r"\begin{tabular}{llrr}",r"\toprule",
           r"Workload & Policy & Energy (mJ) & Delay (ms) \\",r"\midrule"]
    for wl in [w for w in WORKLOADS if (w,"Baseline") in es.index]:
        for j,p in enumerate(POLICY_ORDER):
            e=es.loc[(wl,p)]; d=ds.loc[(wl,p)]
            name=wl if j==0 else ""
            lines.append(f"{name} & {p} & {e['mean']:.0f}$\\pm${e['ci95']:.0f} "
                         f"& {d['mean']:.0f}$\\pm${d['ci95']:.0f} \\\\")
        lines.append(r"\midrule")
    lines[-1]=r"\bottomrule"; lines.append(r"\end{tabular}")
    return "\n".join(lines)

(OUT_DIR/"table_results.tex").write_text(latex_results())
print((OUT_DIR/"table_results.tex").read_text()[:600])
print("\nWrote:", OUT_DIR/"table_results.tex")

## 8. Data-collection templates (hardware side)

These are **scaffolds** for gathering the raw runs. They contain no
device secrets and perform no measurement until you fill in the
platform-specific hooks. Two paths are provided:

- **A. Instrumented device + power meter** — drive the workload, read the
  meter, log per-run energy/latency to `data/raw_runs.csv`.
- **B. Trace replay** — feed recorded event traces to the compiled
  `build/uatimer_replay` to check control behaviour (transition counts,
  sleep time). This does **not** produce energy values.

In [ ]:
# --- Template A: measurement loop (fill in the TODO hooks) -----------------
import csv, subprocess, time

def measure_energy_mj(platform, workload, policy, run_idx):
    """TODO: return measured energy (mJ) for one run.
    Integrate your Monsoon / external-meter reading over the run window.
    Must return a real measurement; do not synthesize."""
    raise NotImplementedError("wire up the power meter here")

def measure_delay_ms(platform, workload, policy, run_idx):
    """TODO: return mean event delay (ms) from timestamp logs."""
    raise NotImplementedError("wire up latency logging here")

def collect(platform, workloads, policies, n_runs, out_csv):
    new = not Path(out_csv).exists()
    with open(out_csv,"a",newline="") as f:
        w=csv.writer(f)
        if new: w.writerow(REQUIRED_COLS)
        for wl in workloads:
            for pol in policies:
                for r in range(1,n_runs+1):
                    e=measure_energy_mj(platform,wl,pol,r)
                    d=measure_delay_ms(platform,wl,pol,r)
                    w.writerow([platform,wl,pol,r,e,d,"",""])
                    print(f"{platform} {wl} {pol} run {r}: {e:.1f} mJ, {d:.1f} ms")

# Example (disabled until the hooks above are implemented):
# collect("phone", WORKLOADS, POLICY_ORDER, 10, DATA_CSV)
print("Template A ready. Implement the two hooks, then call collect().")

In [ ]:
# --- Template B: trace replay via the compiled harness ---------------------
def replay(trace, config):
    exe = ROOT/"build"/"uatimer_replay"
    if not exe.exists():
        print("build first: run `make` in the repo root"); return None
    out = subprocess.run([str(exe), str(trace), str(config)],
                         capture_output=True, text=True)
    print(out.stdout or out.stderr)
    return out.stdout

# Example:
# replay(ROOT/"traces"/"web.trace", ROOT/"config"/"mobile.cfg")
print("Template B ready. Build the harness, then call replay(trace, config).")

## Summary

- Fill `data/raw_runs.csv` with real per-run measurements (≥10 runs/cell),
  set `USE_SYNTHETIC_DEMO = False`, and re-run all cells.
- The aggregation, CI, significance, figures, and LaTeX tables regenerate
  automatically from your data.
- Copy `results/table_results.tex` / `table_reductions.tex` and the figures
  into the manuscript, replacing the current hand-entered Table V/VI.

This notebook fabricates nothing: with the demo flag off it only reports
what your measurements contain.